# 🔑 Part 0: Google Cloud Authentication
Run this first. Click through the Google permissions popup.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Authenticated successfully!")

# ⚡ Part 1: Install Dependencies

In [ ]:
!pip install segmentation-models-pytorch rasterio

# 🌩️ Part 2: Download Sen1Floods11 Dataset
Pulls ~446 hand-labeled Sentinel-1 flood scenes directly into Colab.

In [ ]:
!mkdir -p /content/sen1floods11/HandLabeled
!gcloud storage cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/* /content/sen1floods11/HandLabeled/

# 📦 Part 3: Imports & GPU Check

In [ ]:
import os, glob, random
import torch
import rasterio
import numpy as np
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
import segmentation_models_pytorch as smp
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Running on: {device}")

# 📊 Part 4: Dataset — with Augmentation + 72/18/10 Split

Two changes from the previous version:
1. **Random horizontal + vertical flips** applied to the training set only.
2. **72% train / 18% val / 10% test split** — the 10% test set is held-out and only used once at the end for final reported metrics.

In [ ]:
class Sen1FloodsDataset(Dataset):
    def __init__(self, data_dir):
        self.image_paths = sorted(glob.glob(
            os.path.join(data_dir, "**", "*_S1Hand.tif"), recursive=True
        ))
        self.mask_paths = [
            p.replace("S1Hand", "LabelHand") for p in self.image_paths
        ]
        print(f"✅ Found {len(self.image_paths)} image/mask pairs.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with rasterio.open(self.image_paths[idx]) as src:
            image = src.read()
            image = np.nan_to_num(image)
            image = np.clip(image, -30, 0)
            image = (image + 30) / 30.0
        with rasterio.open(self.mask_paths[idx]) as src:
            mask = src.read(1)
            mask = np.where(mask == 1, 1.0, 0.0)
        image_tensor = torch.tensor(image, dtype=torch.float32)
        mask_tensor  = torch.tensor(mask,  dtype=torch.float32).unsqueeze(0)
        return image_tensor, mask_tensor


class AugmentedSubset(Dataset):
    """Wraps a Subset and applies random flips on-the-fly."""
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        image, mask = self.subset[idx]
        if random.random() > 0.5:
            image = TF.hflip(image)
            mask  = TF.hflip(mask)
        if random.random() > 0.5:
            image = TF.vflip(image)
            mask  = TF.vflip(mask)
        return image, mask


# ── 72 / 18 / 10 split (fixed seed for reproducibility) ────────────────────
full_dataset = Sen1FloodsDataset("/content/sen1floods11/HandLabeled/")

total      = len(full_dataset)
test_size  = int(0.10 * total)              # ~45 — never touched during training
val_size   = int(0.18 * total)              # ~80 — used for LR scheduling
train_size = total - test_size - val_size   # ~321

train_base, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = AugmentedSubset(train_base)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# 🧠 Part 5: Model + Combined Dice + BCE Loss

**Key upgrade:** replace the old `BCEWithLogitsLoss` with `DiceLoss + SoftBCEWithLogitsLoss`.

- **DiceLoss** directly optimises mask overlap — far better for flood segmentation where flood pixels are rare.
- **SoftBCEWithLogitsLoss** keeps `pos_weight=10` to heavily penalise missed floods.
- Combined loss = best of both. Expect Val IoU to jump from ~0.46 → 0.55+.

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=2,
    classes=1,
).to(device)

# ── Combined loss ────────────────────────────────────────────────────────────
dice_loss = smp.losses.DiceLoss(mode="binary")
bce_loss  = smp.losses.SoftBCEWithLogitsLoss(
    pos_weight=torch.tensor([10.0]).to(device)
)

def criterion(outputs, masks):
    return dice_loss(outputs, masks) + bce_loss(outputs, masks)

# ── Optimizer + scheduler ────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5
)

print("⚖️  Model + Combined Dice+BCE loss initialised.")

# 📈 Part 6: Metric Helpers

In [ ]:
def iou_score(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

def dice_score(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred * target).sum()
    return (2 * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)

print("✅ Metric helpers ready.")

# 🚀 Part 7: Training Loop — 30 Epochs

In [ ]:
def train_model(model, train_loader, val_loader, epochs=30):
    print("🚀 Starting training...")
    best_val_loss = float("inf")

    for epoch in range(epochs):

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        t_loss = t_iou = t_dice = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            t_iou  += iou_score(outputs, masks).item()
            t_dice += dice_score(outputs, masks).item()
        n_tr = len(train_loader)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        v_loss = v_iou = v_dice = 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)
                v_loss += criterion(outputs, masks).item()
                v_iou  += iou_score(outputs, masks).item()
                v_dice += dice_score(outputs, masks).item()
        n_val = len(val_loader)
        avg_val_loss = v_loss / n_val
        scheduler.step(avg_val_loss)

        # ── Save best ─────────────────────────────────────────────────────────
        tag = ""
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "/content/flood_unet_resnet34_best.pth")
            tag = "  ⭐ BEST SAVED"

        lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train Loss: {t_loss/n_tr:.4f} | "
            f"Train IoU: {t_iou/n_tr:.4f} | Train Dice: {t_dice/n_tr:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Val IoU: {v_iou/n_val:.4f} | Val Dice: {v_dice/n_val:.4f} | "
            f"LR: {lr:.6f}{tag}"
        )

    print(f"\n✅ Training complete. Best val loss: {best_val_loss:.4f}")


train_model(model, train_loader, val_loader, epochs=30)

# 🎯 Part 8: Final Test Set Evaluation

Load the **best saved checkpoint** and evaluate on the **held-out 10% test set**.
These are the numbers you report in your paper/presentation — they were never seen during training or hyperparameter tuning.

In [ ]:
# Load the best checkpoint
model.load_state_dict(torch.load("/content/flood_unet_resnet34_best.pth", map_location=device))
model.eval()

test_iou = test_dice = 0.0
with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(images)
        test_iou  += iou_score(outputs, masks).item()
        test_dice += dice_score(outputs, masks).item()

n_test = len(test_loader)
print("=" * 50)
print(f"  FINAL TEST SET RESULTS  ({len(test_dataset)} held-out samples)")
print(f"  Test IoU  : {test_iou  / n_test:.4f}")
print(f"  Test Dice : {test_dice / n_test:.4f}")
print("=" * 50)
print("  Use these numbers in your report.")

# 📸 Part 9: Visualize a Validation Prediction

In [ ]:
def visualize_prediction(model, dataset, index=0):
    model.eval()
    image, true_mask = dataset[index]
    with torch.no_grad():
        pred = torch.sigmoid(model(image.unsqueeze(0).to(device)))
        pred_mask = (pred > 0.5).float().cpu().numpy().squeeze()

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image[0].numpy(), cmap="gray");             axes[0].set_title("📡 SAR VV Band");      axes[0].axis("off")
    axes[1].imshow(true_mask.numpy().squeeze(), cmap="Blues"); axes[1].set_title("🗺️  True Mask");       axes[1].axis("off")
    axes[2].imshow(pred_mask, cmap="Blues");                   axes[2].set_title("🤖 Predicted Mask");  axes[2].axis("off")
    plt.tight_layout(); plt.show()

visualize_prediction(model, val_dataset, index=15)

# 📥 Part 10: Download the Best Model

Downloads `flood_unet_resnet34.pth` to your computer.
Drop it into `disaster-ai-api/` replacing the old one, then restart `python main.py`.

In [ ]:
import shutil
from google.colab import files

shutil.copy("/content/flood_unet_resnet34_best.pth", "/content/flood_unet_resnet34.pth")
print("✅ Saved as flood_unet_resnet34.pth")
files.download("/content/flood_unet_resnet34.pth")
print("📥 Download triggered. Replace disaster-ai-api/flood_unet_resnet34.pth with this file.")

# 🧪 Part 11: (Optional) Export a Real Test TIF

Export a real Sentinel-1 scene from the test set to verify the new model via the dashboard Upload tab.

In [ ]:
sample_num = 5   # change to whichever test sample looks best in visualize_prediction

subset_index = test_dataset.indices[sample_num]
real_image_path = full_dataset.image_paths[subset_index]
print(f"✅ Exporting: {real_image_path}")

with rasterio.open(real_image_path) as src:
    data = src.read()
    meta = src.meta

with rasterio.open("/content/real_flood_test_v2.tif", "w", **meta) as dst:
    dst.write(data)

files.download("/content/real_flood_test_v2.tif")
print("📥 TIF downloaded. Use it in the Upload TIF tab to test the new model.")